# Lab for Week 2

## Implementation of Logicstic Regression

You will implement a Logistic regression by modifying/extending the Pytorch (concise) code of Linear regression, and apply it to two different datasets. 

Briefly, Logistic regression (or Logit regression) model is another class of regression models that is used to estimate the probability that a given instance of a dataset belongs to a particular class. You may think of a classic example of spam and non-spam emails. The idea is that if the estimated probality is greater than 50%, then the model predicts that the instance belongs to a positve class, say 'A', else it predicts that it belongs to negative class, B. This definition makes the Logistic regression model a binary classifier.

### Overview of the problem

Suppose that you are the administrator of a university department and you want to determine each applicant's chance of admission based on their results on two exams. You have historical data from previous applicants that you can use as a training set for logistic regression. For each training example, you have the applicant's scores on two exams and the admissions decision.

Your task is to build a classification model that estimates an applicant's probability of admission based the scores from those two exams.

Therefore, you will build a logistic regression model to predict whether a student gets admitted into a university.

In [34]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import jax_dataloader as jdl
from jax import numpy as jnp
import torch
from flax import nnx
import optax

In [35]:
class ExData(Dataset):
    def __init__(self, file):
        self.data = np.loadtxt(file, delimiter=',', dtype=np.float32)
        # Simple normalization: (x - mean) / std
        self.data[:, 0:2] = (self.data[:, 0:2] - self.data[:, 0:2].mean(axis=0)) / self.data[:, 0:2].std(axis=0)
        print(self.data.shape)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx, 0:2], self.data[idx, 2:3]

train_data = ExData("./ex2data1.txt")
train_dataloader = jdl.DataLoader(train_data, "pytorch", batch_size=32, shuffle=True, to_jax=True)

(100, 3)


In [ ]:
Rngs = nnx.Rngs(0)
model = nnx.Linear(2, 1, rngs=Rngs)

tx = optax.sgd(0.03)
optimiser = nnx.Optimizer(model, tx, wrt=nnx.Param)

def forward(model: nnx.Module, features: np.ndarray, labels: np.ndarray):
    pred = model(features)
    loss = optax.sigmoid_binary_cross_entropy(pred, labels).mean()
    return loss

@nnx.jit
def train_step(model: nnx.Module, optimiser: nnx.Optimizer, features, labels):
    loss, grad = nnx.value_and_grad(forward)(model, features, labels)
    optimiser.update(model, grad)
    return loss

num_epochs = 1000
for epoch in range(num_epochs):
    for X, y in train_dataloader:
        loss = train_step(model, optimiser, X, y)

print(model.kernel, model.bias)


loss=0.803526759147644
loss=0.26527827978134155
loss=0.2943231463432312
loss=0.20323708653450012
loss=0.2024724781513214
loss=0.2825794816017151
loss=0.2728458344936371
loss=0.16773134469985962
loss=0.2469252645969391
loss=0.21052402257919312
Param( # 2 (8 B)
  value=Array([[3.197064],
         [2.942569]], dtype=float32)
) Param( # 1 (4 B)
  value=Array([1.3193098], dtype=float32)
)


In [ ]:
Rngs = nnx.Rngs(0)
model = nnx.Linear(2, 1, rngs=Rngs)
tx = optax.sgd(0.03)
optimiser = nnx.Optimizer(model, tx, wrt=nnx.Param)

def forward(model: nnx.Module, features: np.ndarray, labels: np.ndarray):
    pred = model(features)
    loss = optax.sigmoid_binary_cross_entropy(pred, labels).mean()
    return loss

@nnx.jit
def train_step(model: nnx.Module, optimiser: nnx.Optimizer, features, labels):
    loss, grad = nnx.value_and_grad(forward)(model, features, labels)
    optimiser.update(model, grad)
    return loss

num_epochs = 1000
for epoch in range(num_epochs):
    avg_loss = 0
    n = 1
    for X, y in train_dataloader:
        loss = train_step(model, optimiser, X, y)
        avg_loss = avg_loss + (loss - avg_loss)/n
        n += 1
    
    if epoch%100 == 0:
        print(f"loss={avg_loss}")
    
print(model.kernel, model.bias)
